In [1]:
import numpy as np
import astropy.units as u
import datetime
import glob
import pandas as pd

In [2]:
#Read in CME data
files = glob.glob("Data/Donki/*")
df = pd.DataFrame()
for i in files:
    temp = pd.read_csv(i)
    df = pd.concat([df,temp],ignore_index=True)
    
df_CME = pd.DataFrame()
df_CME["CME_Time"] = pd.to_datetime(df["5"].str[1:-8],format="%Y-%m-%dT%H:%M:%S").astype("datetime64[s]")

# #Correct a typo
dtimes = df["0"].to_numpy()
dtimes[df["0"].str[0] == "0"] = "2" + np.array(df["0"][df["0"].str[0] == "0"])[0][1:]
df["0"] = dtimes

#Format the data properly 
df_CME["Time_21.5"] = pd.to_datetime(df["0"],format="%Y-%m-%dT%H:%M")
df_CME["lat"] = df["1"].str.slice(4).astype(float)
df_CME["lon"] = df["2"].str.slice(4).astype(float)
df_CME["Ang_rad"] = df["3"].str.slice(4).astype(float)
df_CME["V"] = df["4"].str.slice(4).astype(float)
df_CME["cat"] = df["6"]

#Set index to CME time
df_CME = df_CME.set_index("CME_Time")

#Remove duplicate entries which are corresponding to the same CME - shock vs leading edge
df_CME = df_CME.drop(df_CME.index[df_CME.index.duplicated(keep=False)][(df["8"][df_CME.index.duplicated(keep=False)]=="SH")+(pd.isnull(df["8"][df_CME.index.duplicated(keep=False)]))])
#Remove duplicate entries from different source catalogues 
df_CME = df_CME.drop(df_CME["cat"][df_CME.index.duplicated(keep=False)][df_CME["cat"][df_CME.index.duplicated(keep=False)] != "M2M_CATALOG"].index)
#Remove duplicate entries which result from different model runs
df_CME = df_CME.drop(df_CME.index[df_CME.duplicated()])

#Sort by time at 21.5 Rsun
df_CME = df_CME.sort_values("Time_21.5")

In [3]:
#Read in ICME data

####### Change this to include additional columns #########
col_numbers = [0,1,2,11,12,13,14]
col_names = ["Disturbance_Time","ICME_Start_Time","ICME_End_Time","V_ICME","V_max","B","MC"]
###########################################################

#Read in data
df_I = pd.read_csv("Data/ICME_RichardsonCane.csv",usecols=col_numbers,names=col_names,skiprows=1)
df_I["ICME_Start_Time"] = pd.to_datetime(df_I["ICME_Start_Time"],format="%Y/%m/%d %H%M")
df_I["ICME_End_Time"] = pd.to_datetime(df_I["ICME_End_Time"],format="%Y/%m/%d %H%M")

In [4]:
#Read in Omni Data
filepath = "Data/Solar Wind/Omni/omni_1hour.h5"
omni_data_df = pd.read_hdf(filepath)
omni_data_df = omni_data_df.set_index("datetime")

df_Wind = pd.DataFrame()
df_Wind["DateTime"] = omni_data_df.index
df_Wind["V_wind"] = omni_data_df.V.to_numpy()
df_Wind = df_Wind.set_index("DateTime")

In [5]:
#Algorithm to odentify the CMEs associated with each ICME 

#Decimal Error in CME speed, and angular size of CME - 0.1,0.1 tested to be good values
speed_err = 0.1
ang_err = 0.1

#Set up outputs
Single_events = []
Multi_events = []

#Iterate through each ICME
for ind in range(len(df_I)):
    #Find time of ICME at Earth
    t_ICME = df_I.ICME_Start_Time.to_numpy()[ind]

    #Find CMEs occuring within the past 6 days
    CME_subset = pd.DataFrame(df_CME[(df_CME["Time_21.5"]>t_ICME-np.timedelta64(6,"D")) & (df_CME["Time_21.5"]<t_ICME)])

    #Specify position of valid CMEs
        #Angular distance of CME from (0,0)
    ang_distance = np.arccos( np.cos(CME_subset.lat*np.pi/180) * np.cos(CME_subset.lon*np.pi/180)) * 180/np.pi
        #1 if the angular distance < (CME angular size + error), 0 otherwise
    good_pos = (ang_distance < (1+ang_err)*CME_subset.Ang_rad).to_numpy()

    #Specify appropriate speeds/timings
        #Speed of CME at Sun
    v_cme = CME_subset.V.to_numpy()
        #Speed recorded at Earth during ICME 
    Wind_reduced = df_Wind[(df_Wind.index>=df_I.ICME_Start_Time.iloc[ind])&(df_Wind.index<df_I.ICME_End_Time.iloc[ind])]
    v_icme_min = np.min(Wind_reduced)
    v_icme_max = np.max(Wind_reduced)

    #Determine the shortest and longest possible travel times for the CME 
    distance = (149.597e6 - 21.5*696340)
    t_min = distance/ np.max(np.vstack([(1+speed_err)*v_cme,v_icme_max*np.ones(len(v_cme))]),axis=0)
    t_max = distance/ np.min(np.vstack([(1-speed_err)*v_cme,v_icme_min*np.ones(len(v_cme))]),axis=0)

    #Determine the time between the CME emission, and its arrival at Earth
    t_delay = (df_I.ICME_Start_Time.iloc[ind] - CME_subset["Time_21.5"]).to_numpy()/np.timedelta64(1,"s")

    #Specify which CMEs were travelling with appropriate speeds to reach the Earth at the required time 
        #1 if minimum travel time < true travel time < maximum travel time
    good_speed = (t_delay>=t_min)&(t_delay<=t_max)

    
    #Reduce list to filtered criteria 
    CME_filtered = CME_subset[(good_pos==True)&(good_speed==True)]

    #If only 1 valid CME remains, then identify this as an event
    if len(CME_filtered) == 1:
    #Check that a slow CME does not interfere
        #Find CMEs passing 21.5 before the ICME CME does 
        earlier_CMEs = CME_subset["Time_21.5"].to_numpy()<CME_filtered["Time_21.5"].iloc[0]
        #Find if any of these CMEs are unable to reach Earth before the ICME does
        slow = np.any(t_delay[earlier_CMEs & good_pos]<t_min[earlier_CMEs & good_pos])
        if slow:
            Multi_events.append([df_I.ICME_Start_Time.loc[ind],np.array(CME_filtered.index.astype(str))])
        else:
            Single_events.append([df_I.ICME_Start_Time.loc[ind],CME_filtered.index[0]])

    #If more than 1 valid CME remains, then list the events
    if len(CME_filtered) > 1:
        Multi_events.append([df_I.ICME_Start_Time.loc[ind],np.array(CME_filtered.index.astype(str))])
        
    
Single_events = pd.DataFrame(Single_events,columns=["ICME","CME"])
Multi_events = pd.DataFrame(Multi_events,columns=["ICME","CME"])

# Set up outputs
dash = np.zeros(len(Single_events)).astype(str); dash[:] = "-"
dash = pd.DataFrame(dash,columns=["-"])

ICME_indices = np.array([np.where(df_I["ICME_Start_Time"] == ICME)[0][0] for ICME in Single_events.ICME])
ICME_data = df_I.iloc[ICME_indices].reset_index(drop=True)

CME_indices = np.array([np.where(df_CME.index == CME)[0][0] for CME in Single_events.CME])
CME_data = df_CME.iloc[CME_indices].reset_index(drop=False)

Single_events = Single_events.join(dash,how="inner")
Single_events = Single_events.join(ICME_data,how="inner")
Single_events = Single_events.join(dash,how="inner",rsuffix=" ")
Single_events = Single_events.join(CME_data,how="inner")

In [6]:
# #Output data to csvs
Single_events.to_csv("(I)CMEs_SingleEvents.csv",index=False)
Multi_events.to_csv("(I)CMEs_MultiEvents.csv",index=False)